# Лабораторная работа: Реализация CART с нуля

Датасет UCI Rice — Cammeo vs Osmancik.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.io import arff
from dataclasses import dataclass
from typing import Optional
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
DATA_PATH = Path("data/rice/raw/Rice_Cammeo_Osmancik.arff")

raw_data, meta = arff.loadarff(DATA_PATH)
df_raw = pd.DataFrame(raw_data)
df_raw["Class"] = df_raw["Class"].str.decode("utf-8")

print("Первые 5 меток:", df_raw["Class"].head().tolist())
print("Последние 5 меток:", df_raw["Class"].tail().tolist())


## Анализ упорядоченности

Исходный ARFF-файл сгруппирован по сортам. Поэтому обычное разбиение по срезам строк может привести к попаданию одного класса только в одну часть выборки. Перед разбиением используется случайное перемешивание.

In [ ]:
df = df_raw.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
feature_names = [c for c in df.columns if c != "Class"]
target_col = "Class"

assert df.shape == (3810, 8)
assert len(feature_names) == 7
assert df[feature_names].isna().sum().sum() == 0
assert set(df[target_col].unique()) == {"Cammeo", "Osmancik"}
assert target_col not in feature_names
print("Инварианты загрузки успешно подтверждены.")


## Раздел 1. Разведочный анализ и честное разбиение

In [ ]:
print(df[feature_names].describe().T[["count","mean","std","min","50%","max"]])
print("\nДоли классов:\n", df["Class"].value_counts(normalize=True))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()
for i, col in enumerate(feature_names):
    axes[i].hist(df[col], bins=30, edgecolor="black", alpha=0.7)
    axes[i].set_title(col)
fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,5))
for label in ["Cammeo", "Osmancik"]:
    m = df["Class"] == label
    plt.scatter(df.loc[m, "Area"], df.loc[m, "Convex_Area"], label=label, alpha=0.4, s=15)
plt.xlabel("Area")
plt.ylabel("Convex_Area")
plt.title("Геометрическая связь Area и Convex_Area")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


Классы умеренно сбалансированы: примерно 57% Osmancik и 43% Cammeo. Area и Convex_Area имеют сильную геометрическую связь, поэтому их важности в дереве могут перераспределяться.

In [ ]:
class_mapping = {"Cammeo": 0, "Osmancik": 1}
inv_class_mapping = {0: "Cammeo", 1: "Osmancik"}
y_all = df["Class"].map(class_mapping).values
X_all = df[feature_names].values

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.20, stratify=y_train_val, random_state=RANDOM_STATE
)

print(f"Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")


## Раздел 2. Критерии и кандидатные разбиения

Для узла из 6 объектов с распределением 4/2: Gini = 4/9 ≈ 0.4444, Entropy ≈ 0.9183 бит.

In [ ]:
def class_counts(y, n_classes=2):
    if len(y) == 0:
        return np.zeros(n_classes, dtype=np.int64)
    return np.bincount(y, minlength=n_classes)

def gini_from_counts(counts):
    c = np.asarray(counts, dtype=float)
    if c.ndim == 1:
        total = c.sum()
        if total == 0:
            return 0.0
        p = c / total
        return 1.0 - np.sum(p ** 2)
    totals = c.sum(axis=1, keepdims=True)
    safe = np.where(totals == 0, 1.0, totals)
    p = c / safe
    return np.where(totals.ravel() == 0, 0.0, 1.0 - np.sum(p ** 2, axis=1))

def entropy_from_counts(counts):
    c = np.asarray(counts, dtype=float)
    if c.ndim == 1:
        total = c.sum()
        if total == 0:
            return 0.0
        p = c[c > 0] / total
        return -np.sum(p * np.log2(p))
    totals = c.sum(axis=1, keepdims=True)
    safe = np.where(totals == 0, 1.0, totals)
    p = c / safe
    ps = np.where(p > 0, p, 1.0)
    e = -np.sum(np.where(p > 0, p * np.log2(ps), 0.0), axis=1)
    return np.where(totals.ravel() == 0, 0.0, e)

def candidate_thresholds(values):
    u = np.unique(values)
    return np.empty(0, dtype=float) if len(u) < 2 else (u[:-1] + u[1:]) / 2.0

def weighted_impurity(left, right, criterion):
    f = gini_from_counts if criterion == "gini" else entropy_from_counts
    hl, hr = f(left), f(right)
    nl, nr = left.sum(axis=-1), right.sum(axis=-1)
    total = nl + nr
    wl = np.where(total == 0, 0, nl / total)
    wr = np.where(total == 0, 0, nr / total)
    return wl * hl + wr * hr

def best_split(X, y, n_classes, criterion, min_samples_leaf):
    parent = class_counts(y, n_classes)
    parent_imp = gini_from_counts(parent) if criterion == "gini" else entropy_from_counts(parent)
    best_gain, best_j, best_t, best_mask = 0.0, None, None, None

    for j in range(X.shape[1]):
        values = X[:, j]
        thresholds = candidate_thresholds(values)
        if len(thresholds) == 0:
            continue

        idx = np.argsort(values)
        xs, ys = values[idx], y[idx]
        positions = np.searchsorted(xs, thresholds, side="right")
        one_hot = np.eye(n_classes)[ys]
        cumulative = np.cumsum(one_hot, axis=0)
        left_counts = cumulative[positions - 1]
        right_counts = parent - left_counts
        nl, nr = positions, len(y) - positions
        valid = (nl >= min_samples_leaf) & (nr >= min_samples_leaf)

        if not np.any(valid):
            continue

        gains = parent_imp - weighted_impurity(
            left_counts[valid], right_counts[valid], criterion
        )
        k = np.argmax(gains)

        if gains[k] > best_gain:
            vt = thresholds[valid]
            best_gain = gains[k]
            best_j = j
            best_t = vt[k]
            best_mask = values <= best_t

    return best_j, best_t, best_gain, best_mask

assert gini_from_counts(np.array([10,0])) == 0.0
assert entropy_from_counts(np.array([0,10])) == 0.0
assert np.isclose(gini_from_counts(np.array([5,5])), 0.5)
assert np.isclose(entropy_from_counts(np.array([5,5])), 1.0)
print("Инварианты критериев успешно проверены.")


## Раздел 3. Узел и рекурсивное дерево

In [ ]:
@dataclass
class TreeNode:
    depth: int
    n_samples: int
    class_counts: np.ndarray
    prediction: int
    probabilities: np.ndarray
    impurity: float
    feature_index: Optional[int] = None
    threshold: Optional[float] = None
    left: Optional["TreeNode"] = None
    right: Optional["TreeNode"] = None

    @property
    def is_leaf(self):
        return self.left is None and self.right is None

class CARTClassifier:
    def __init__(self, criterion="gini", max_depth=None,
                 min_samples_split=2, min_samples_leaf=1,
                 min_impurity_decrease=0.0):
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.min_impurity_decrease = min_impurity_decrease
        self.root = None
        self.classes_ = None
        self.n_features_ = None
        self.feature_importances_ = None

    def _impurity(self, counts):
        return gini_from_counts(counts) if self.criterion == "gini" else entropy_from_counts(counts)

    def _build_tree(self, X, y, depth, raw_importances):
        n = len(y)
        counts = class_counts(y, len(self.classes_))
        prediction = int(np.argmax(counts))
        probabilities = counts / n
        impurity = self._impurity(counts)

        stop = (
            impurity == 0.0 or
            (self.max_depth is not None and depth >= self.max_depth) or
            n < self.min_samples_split or
            n < 2 * self.min_samples_leaf
        )
        if stop:
            return TreeNode(depth, n, counts, prediction, probabilities, impurity)

        j, t, gain, mask = best_split(
            X, y, len(self.classes_), self.criterion, self.min_samples_leaf
        )
        if j is None or gain < self.min_impurity_decrease:
            return TreeNode(depth, n, counts, prediction, probabilities, impurity)

        raw_importances[j] += n * gain
        left = self._build_tree(X[mask], y[mask], depth + 1, raw_importances)
        right = self._build_tree(X[~mask], y[~mask], depth + 1, raw_importances)

        return TreeNode(depth, n, counts, prediction, probabilities, impurity,
                        j, t, left, right)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)
        self.classes_ = np.unique(y)
        self.n_features_ = X.shape[1]
        raw = np.zeros(self.n_features_, dtype=float)
        self.root = self._build_tree(X, y, 0, raw)
        total = raw.sum()
        self.feature_importances_ = raw / total if total > 0 else raw
        return self

    def _predict_row(self, x, node):
        current = node
        while not current.is_leaf:
            current = current.left if x[current.feature_index] <= current.threshold else current.right
        return current.prediction, current.probabilities

    def predict_proba(self, X):
        return np.array([self._predict_row(row, self.root)[1] for row in np.asarray(X, dtype=float)])

    def predict(self, X):
        indices = [self._predict_row(row, self.root)[0] for row in np.asarray(X, dtype=float)]
        return self.classes_[np.array(indices)]


## Раздел 4. Контролируемый toy-тест

In [ ]:
X_toy = np.array([[1.0],[2.0],[3.0],[7.0],[8.0],[9.0]])
y_toy = np.array([0,0,0,1,1,1])

for criterion in ["gini", "entropy"]:
    clf = CARTClassifier(criterion=criterion, max_depth=1).fit(X_toy, y_toy)
    assert clf.root.feature_index == 0
    assert 3.0 < clf.root.threshold < 7.0
    assert np.all(clf.predict(X_toy) == y_toy)

X_const = np.array([[5.0],[5.0],[5.0],[5.0]])
y_const = np.array([0,0,1,1])
assert CARTClassifier(max_depth=2).fit(X_const, y_const).root.is_leaf

assert CARTClassifier(max_depth=2, min_samples_leaf=4).fit(X_toy, y_toy).root.is_leaf
print("Все модульные toy-тесты пройдены.")


## Раздел 5. Первое дерево и визуализация

In [ ]:
base_cart = CARTClassifier(
    criterion="gini", max_depth=3, min_samples_leaf=10
).fit(X_train, y_train)

def tree_depth(node):
    if node.is_leaf:
        return node.depth
    return max(tree_depth(node.left), tree_depth(node.right))

print("Глубина дерева:", tree_depth(base_cart.root))
print("Важность признаков:")
print(pd.DataFrame({"Feature": feature_names,
                    "Importance": base_cart.feature_importances_}))

val_preds = base_cart.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print("Validation Balanced Accuracy:", balanced_accuracy_score(y_val, val_preds))
print("Confusion Matrix:\n", confusion_matrix(y_val, val_preds))


## Раздел 6. Выбор гиперпараметров по validation

In [ ]:
grid_criterions = ["gini", "entropy"]
grid_max_depths = [2, 3, 4, 5, 6, None]
grid_min_samples_leaf = [5, 10, 20, 40]
results = []

def tree_stats(node):
    if node.is_leaf:
        return 1, node.depth
    a, da = tree_stats(node.left)
    b, db = tree_stats(node.right)
    return a + b, max(da, db)

for criterion in grid_criterions:
    for depth in grid_max_depths:
        for min_leaf in grid_min_samples_leaf:
            model = CARTClassifier(
                criterion=criterion,
                max_depth=depth,
                min_samples_split=2 * min_leaf,
                min_samples_leaf=min_leaf
            ).fit(X_train, y_train)

            train_pred = model.predict(X_train)
            val_pred = model.predict(X_val)
            leaves, actual_depth = tree_stats(model.root)

            results.append({
                "criterion": criterion,
                "max_depth": depth,
                "min_samples_leaf": min_leaf,
                "leaves": leaves,
                "depth": actual_depth,
                "train_acc": accuracy_score(y_train, train_pred),
                "val_acc": accuracy_score(y_val, val_pred),
                "val_bacc": balanced_accuracy_score(y_val, val_pred)
            })

df_grid = pd.DataFrame(results).sort_values(
    ["val_acc", "val_bacc", "depth"],
    ascending=[False, False, True]
).reset_index(drop=True)

display(df_grid.head())
best_config = df_grid.iloc[0].to_dict()
print("Лучшая конфигурация:", best_config)


## Раздел 7. Финальная оценка и сопоставление со sklearn

In [ ]:
best_cart = CARTClassifier(
    criterion=best_config["criterion"],
    max_depth=int(best_config["max_depth"]) if best_config["max_depth"] is not None else None,
    min_samples_split=int(2 * best_config["min_samples_leaf"]),
    min_samples_leaf=int(best_config["min_samples_leaf"])
).fit(X_train_val, y_train_val)

sk_cart = DecisionTreeClassifier(
    criterion=best_config["criterion"],
    max_depth=int(best_config["max_depth"]) if best_config["max_depth"] is not None else None,
    min_samples_split=int(2 * best_config["min_samples_leaf"]),
    min_samples_leaf=int(best_config["min_samples_leaf"]),
    random_state=RANDOM_STATE
).fit(X_train_val, y_train_val)

y_pred_my = best_cart.predict(X_test)
y_pred_sk = sk_cart.predict(X_test)

print("Custom CART:",
      accuracy_score(y_test, y_pred_my),
      balanced_accuracy_score(y_test, y_pred_my))
print("Scikit-Learn:",
      accuracy_score(y_test, y_pred_sk),
      balanced_accuracy_score(y_test, y_pred_sk))

print(classification_report(
    y_test, y_pred_my, target_names=["Cammeo", "Osmancik"], digits=4
))

display(pd.DataFrame({
    "Feature": feature_names,
    "Custom_MDI": best_cart.feature_importances_,
    "Sklearn_MDI": sk_cart.feature_importances_
}))


## Раздел 8. Модельная карточка

- Задача: бинарная классификация сортов риса Cammeo и Osmancik.
- Источник: UCI Rice.
- ARFF предварительно перемешан с `random_state=42`.
- Разбиение: Train 64%, Validation 16%, Test 20%.
- Реализованы Gini, Entropy, кандидатные пороги и рекурсивное CART.
- Выполнены toy-тесты и сравнение со scikit-learn.
- Test используется только для финальной оценки.
- Сильная корреляция геометрических признаков может перераспределять MDI.
